# 04_labeling.ipynb - Fase 4: Triple Barrier Labeling

**Objetivo**: Etiquetar cada entrada potencial con {0: SL, 1: TP, 2: Timeout}

Regla de desempate: Si High toca TP Y Low toca SL en misma vela → SL gana (conservador).

In [6]:
import sys
import pandas as pd
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from config.settings import DATA, FEATURES, RISK, BACKTEST
from src.data import load_ohlc_from_yfinance
from src.features import add_all_features
from src.labeling import apply_triple_barrier_to_dataset, create_labeled_dataset

print("Imports listos")

Imports listos


## Labeling para cada ticker admitido

In [7]:
# Ejecutar este bloque con los tickers admitidos
# output: data/labeled_*.csv

admitted_df = pd.read_csv(REPO_ROOT / "data" / "universe_admitted.csv")

for _, row in admitted_df.iterrows():
    ticker = row['ticker']
    csl = row['csl']
    print(f"\nProcesando {ticker} (Csl={csl:.3f})")

    try:
        df = load_ohlc_from_yfinance(ticker, period=DATA.period, interval=DATA.interval)
    except Exception as e:
        print(f"  ❌ Error al cargar {ticker}: {e}")
        continue

    if df.empty:
        print("  ❌ No hay datos disponibles")
        continue

    df = add_all_features(df)
    labeled = create_labeled_dataset(
        df,
        csl,
        tp_sl_ratio=BACKTEST.tp_sl_ratio,
        max_bars=BACKTEST.max_bars
    )

    if labeled.empty:
        print("  ⚠️ No se generaron etiquetas para este ticker")
        continue

    print(f"  ✅ Muestras etiquetadas: {len(labeled)}")
    print(f"  Distribución: {labeled['label'].value_counts().to_dict()}")

    output_path = REPO_ROOT / f"data/labeled_{ticker}.csv"
    labeled.to_csv(output_path, index=False)
    print(f"  Guardado: {output_path}")





Procesando MCRB (Csl=2.073)


  ✅ Muestras etiquetadas: 3637
  Distribución: {0: 1893, 1: 1171, 2: 573}
  Guardado: /home/odoo-01/Escritorio/Projecto_Machine_Learning_Quant/smallcap-quant-ml/data/labeled_MCRB.csv

Procesando LRMR (Csl=2.187)
  ✅ Muestras etiquetadas: 5032
  Distribución: {0: 2754, 1: 1759, 2: 519}
  Guardado: /home/odoo-01/Escritorio/Projecto_Machine_Learning_Quant/smallcap-quant-ml/data/labeled_LRMR.csv

Procesando ABSI (Csl=2.297)
  ✅ Muestras etiquetadas: 5032
  Distribución: {0: 2428, 1: 2057, 2: 547}
  Guardado: /home/odoo-01/Escritorio/Projecto_Machine_Learning_Quant/smallcap-quant-ml/data/labeled_ABSI.csv

Procesando IMUX (Csl=2.348)
  ✅ Muestras etiquetadas: 4997
  Distribución: {0: 2510, 1: 1821, 2: 666}
  Guardado: /home/odoo-01/Escritorio/Projecto_Machine_Learning_Quant/smallcap-quant-ml/data/labeled_IMUX.csv

Procesando KPTI (Csl=2.388)
  ✅ Muestras etiquetadas: 4627
  Distribución: {0: 2277, 1: 1580, 2: 770}
  Guardado: /home/odoo-01/Escritorio/Projecto_Machine_Learning_Quant/smallcap-